# OPX bench demo

Drive the OPX program builder (`kexp.control.opx`) interactively, without an experiment: build the QUA program for a sequence from `kexp/experiments/opx_sequences/`, simulate it on the QOP, and inspect the result.

`OPXBench` stands in for a prepared experiment with **kexp defaults** — the real `ExptParams` and the real dds-frame defaults, so the compiled config is exactly what a run would use — but no ARTIQ, no liveOD, no run id, nothing saved.

**Needs:** the root workspace venv as the kernel, and the OPX reachable on the lab network (the simulator runs on the QOP server, 192.168.1.125).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline
%load_ext autoreload
%autoreload 2

plt.rcParams['figure.dpi'] = 200

from kexp.control.opx import OPXBench
from kexp.experiments.opx_sequences.rabi import rabi_raman_pulse, rabi_raman_apd

bench = OPXBench()

## Simulate one shot of a sequence

Declare xvars exactly as in an experiment's `prepare()`. On the bench, values run **in the order given** (no shuffle, no repeats), so simulated shot $k$ is `values[k]`, deterministically.

`simulate(...)` builds the per-shot tables, traces the program, sends it to the QOP simulator, and pops the waveform report. `shots=1` (default) simulates just the first scheduled shot — every scanned param bakes to that shot's value; `shots=0` simulates the whole scan. `duration` is the simulated OPX timeline in seconds.

In [ ]:
bench.xvar('t_raman_pulse', np.linspace(0., 30.e-6, 15))

job = bench.simulate(rabi_raman_pulse, shots=2, duration=100.e-6)

## Inspect the traces yourself

The waveform report already plotted above. The raw simulated samples are on the job for manual plotting — digital 1 is the raman switch (high = ARTIQ RF blocked), digital 2 the imaging switch, digital 3 the hand-back trigger, analog 1/2 the sticky 80/150 MHz raman drives.

In [ ]:
samples = job.get_simulated_samples()
samples.con1.plot()   # everything on one axis (qm built-in)

In [ ]:
# or by hand, e.g. the two switch lines and the hand-back trigger
fig, ax = plt.subplots(figsize=(8, 2.5))
for port, label in [('1', 'raman switch (1=block)'),
                    ('2', 'imaging switch (1=block)'),
                    ('3', 'hand-back trigger')]:
    d = samples.con1.digital[port]
    ax.plot(np.asarray(d, dtype=float) + 1.2 * float(port), label=label)
ax.set_xlabel('t (ns)')
ax.legend(fontsize=6)
ax.set_title('simulated digital lines (offset for clarity)');

## The generated program

`bench.qua_source` is the **real, triggered** program text — exactly what a run saves into the HDF5 as `opx_qua_program` (the simulation itself runs a variant with `wait_for_trigger` skipped, since the simulator has no trigger source).

In [ ]:
print(bench.qua_source[:2000])

## Tweak params and re-run

Every `simulate()` starts fresh (new manager, new DataVault), so iterate freely: edit a sequence file (autoreload picks it up), change `bench.p.<param>`, re-run the cell. A sequence with `measurements={...}` works the same — the APD acquire windows show up on the ADC marker (digital 4); its DataVault containers exist on the bench but nothing is fetched or saved.

In [ ]:
bench.p.t_raman_pi_pulse = 8.8e-6
job = bench.simulate(rabi_raman_apd, shots=1, duration=400.e-6)

## Define a sequence on the fly

`@opx_sequence` in a notebook cell gives the same object you would import — hand it straight to the bench (or to `self.opx.use(...)` in an experiment). Anything worth keeping should graduate into `kexp/experiments/opx_sequences/`.

In [ ]:
from kexp.control.opx import opx_sequence

@opx_sequence('two_pulse_demo', claims=('raman',))
def two_pulse_demo(ctx):
    """Two Raman exposures separated by a fixed dark time."""
    ctx.raman_pulse(ctx.p.t_raman_pi_pulse)
    ctx.wait_s(5.e-6)
    ctx.raman_pulse(ctx.p.t_raman_pulse)

job = bench.simulate(two_pulse_demo, shots=1, duration=100.e-6)

## In a real experiment

The bench and a run share all machinery; an experiment just swaps the bench for `Base`:

```python
self.opx.use(rabi_raman_pulse)                  # in prepare(), before finish_prepare()
self.finish_prepare(shuffle=True)               # compiles + starts the job
# scan_kernel: prep_raman() -> handoff_to_quantum_machines() -> wait_for_quantum_machines_handoff()
```

Working examples live next to this notebook: `qm_rabi_frequency_opx.py` (camera readout) and `check_rabi_frequency_apd_opx.py` (integrated APD readout). Remember `prep_raman()` before the handoff — the switch-AOM warm-up (shutter closed) is ARTIQ's job, not the OPX's.